# Кейс 1: Рекомендации банковских продуктов  
**Цель:** Предсказать, какие продукты предложить клиентам банка.  
**Основные задачи:** Анализ событий и характеристик продуктов, выбор метрик, построение модели рекомендаций, внедрение модели в виде веб-сервиса, мониторинг и обновление модели.  

---
**Методология (после code review, P3):**
- статистики предобработки (медианы, моды, квантили клиппинга, редкие категории,
  возрастные бины, когорты дат) **обучаются только на train** — без утечки из теста;
- разбиение train/test — **временное** (по дате среза клиента), а не стратифицированное:
  профили одного клиента не смешиваются между частями, метрики честные;
- качество ALS считается на всех покупателях 2016 года (включая cold-start), а не
  только на клиентах, присутствующих в обоих годах.

In [ ]:
# Стандартная библиотека
import gc
import json
import logging
import os
import warnings

# Сторонние библиотеки
from dotenv import load_dotenv
from implicit.als import AlternatingLeastSquares
import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import optuna
import pandas as pd
import scipy
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import tqdm


In [ ]:
gc.collect()


In [ ]:
# Датасет скачивается из соревнования Kaggle ноутбуком loader.ipynb
df = pd.read_csv('data/train_ver2.csv')
df


In [ ]:
# Единый список продуктов и функции предобработки — из общего модуля
# (CODE_REVIEW §P2.14): обучение и сервис используют один и тот же код.
# Код целевой переменной `purchase` = позиция продукта в списке + 1.
from preprocessing import (
    BASE_COLUMNS,
    PRODUCTS as products,
    apply_preprocessing,
    feature_engineering,
    fit_preprocessing_params,
    temporal_split,
)


In [ ]:
# Удаляем малоинформативные столбцы (много пропусков / дублируют другие):
# в старом пайплайне indrel/indext удалялись в начале, nomprov/tipodom — позже;
# все четыре в признаки модели не попадают (см. preprocessing.BASE_COLUMNS).
df = df.drop(columns=['indrel', 'indext', 'nomprov', 'tipodom'])


In [ ]:
# преобразуем столбцы в числовые типы данных
quantitative_cols = ['age', 'antiguedad', 'renta']
for col in quantitative_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# заменяем аномальное значение в antiguedad на NaN
df['antiguedad'] = df['antiguedad'].replace(np.float64(-999999.0), np.nan)

# Преобразуем колонку fecha_dato в формат datetime
df['fecha_dato'] = pd.to_datetime(df['fecha_dato'])


# Персональные
Рассчитаем персональные рекомендации.

In [ ]:
# Целевая переменная строится по всем 24 продуктам, а не по первым трём (CODE_REVIEW §3.1).
# События покупок: продукт появился у клиента впервые (было 0 -> стало 1).
df_sorted = df.sort_values(['ncodpers', 'fecha_dato'])

# Кодируем клиентов и продукты в числовые индексы
users = list(df['ncodpers'].unique())
user_map = {int(user): idx for idx, user in enumerate(users)}
product_map = {product: idx for idx, product in enumerate(products)}

# Предыдущее состояние продукта у клиента — одним groupby на все продукты сразу
products_prev = df_sorted.groupby('ncodpers')[products].shift(1).fillna(0)
current = df_sorted[products].fillna(0).to_numpy(dtype='int8')
previous = products_prev.to_numpy(dtype='int8')
new_purchase = (current == 1) & (previous == 0)

rows, cols = np.nonzero(new_purchase)
events = pd.DataFrame({
    'ncodpers': df_sorted['ncodpers'].to_numpy()[rows],
    'fecha_dato': df_sorted['fecha_dato'].to_numpy()[rows],
    'purchase': (cols + 1).astype(int),  # код продукта = позиция в products + 1
})
print(f'Всего событий покупок: {len(events):,}')
print(events.head())

# Шорт-лист целевых продуктов: берем продукты, на которые пришлось не меньше
# MIN_PURCHASE_SHARE покупок 2015 года (обучающий период, без подглядывания в 2016-й).
# Осознанное сужение задачи: остальные продукты встречаются слишком редко,
# чтобы обучить на них устойчивый класс (CODE_REVIEW §P1.8).
MIN_PURCHASE_SHARE = 0.01   # порог по доле покупок 2015 года
MAX_TARGET_PRODUCTS = 12    # ограничение на размер шорт-листа (размерность классов)
OTHER_CLASS = 99

train_purchases = events[events['fecha_dato'] <= '2016-01-01']
purchase_share = train_purchases['purchase'].value_counts(normalize=True)
candidates = purchase_share[purchase_share >= MIN_PURCHASE_SHARE]
target_codes = sorted(int(code) for code in candidates.index[:MAX_TARGET_PRODUCTS])
dropped_by_cap = [int(code) for code in candidates.index[MAX_TARGET_PRODUCTS:]]

label_map = {0: 'no_purchase'}
label_map.update({code: products[code - 1] for code in target_codes})
label_map[OTHER_CLASS] = 'other'

print(f'Продуктов всего: {len(products)}; прошли порог {MIN_PURCHASE_SHARE:.0%}: '
      f'{len(candidates)}; в шорт-листе: {len(target_codes)}')
for code in target_codes:
    print(f'  {code:>2} = {products[code - 1]:<22} доля покупок 2015: {purchase_share[code]:.2%}')
if dropped_by_cap:
    print(f'Отброшено ограничением MAX_TARGET_PRODUCTS={MAX_TARGET_PRODUCTS}: '
          f'{[products[code - 1] for code in dropped_by_cap]}')
print(f'Класс {OTHER_CLASS} = other: {1 - purchase_share[target_codes].sum():.2%} покупок 2015 '
      f'приходится на продукты вне шорт-листа')
print('label_map:', label_map)


In [ ]:
common_train = events[events['fecha_dato'] <= '2016-01-01'].copy()
common_test = events[events['fecha_dato'] > '2016-01-01'].copy()


In [ ]:
# создаём sparse-матрицу формата CSR
# ВАЖНО: строки матрицы — кодированные индексы user_map, а не сырые ncodpers.
# Раньше в строки попадали сырые ncodpers (1..N), а recommend() вызывался уже с
# user_map-индексами (0..N-1) — показатели строк и вызовы разъезжались, и ALS-recommend
# возвращал рекомендации «соседнего» клиента (off-by-one против user_map).
# Исправлено: rows и обращения в recommend используют одну нумерацию — user_map.
user_item_matrix_train = scipy.sparse.csr_matrix((
    np.ones(len(common_train)),
    (common_train['ncodpers'].map(user_map).to_numpy(),
     common_train['purchase'].to_numpy())),
    dtype=np.int8)


In [ ]:
als_model = AlternatingLeastSquares(factors=50, iterations=50, regularization=0.05,
                                    random_state=0)
als_model.fit(user_item_matrix_train)


In [ ]:
def get_recommendations_als(user_item_matrix, model, user_id, user_map, product_map,
                            include_seen=True, n=5):
    """Возвращает отранжированные ALS-рекомендации для заданного пользователя."""
    # Получаем закодированный user_id
    user_id_enc = user_map[user_id]

    # Создаем обратный словарь для product_map
    inv_product_map = {v: k for k, v in product_map.items()}

    # Получаем рекомендации от модели
    indices, scores = model.recommend(
        user_id_enc,
        user_item_matrix[user_id_enc],
        filter_already_liked_items=not include_seen,
        N=n
    )

    # Создаем итоговый DataFrame в pandas (polars в проекте не используется)
    return pd.DataFrame({
        'product': [inv_product_map.get(int(index)) for index in indices],
        'score': scores
    })


In [ ]:
# Пересечение через set: раньше .unique() пересчитывался на каждой итерации
train_users = common_train['ncodpers'].unique()
test_users = set(common_test['ncodpers'].unique())
common_users = np.array([u for u in train_users if u in test_users])

# user_map — dict, вытаскиваем индексы напрямую
user_ids_encoded = np.array([user_map[int(x)] for x in common_users], dtype=np.int32)

# Батчевые рекомендации: ALS.recommend на списке крутит Python-цикл внутри,
# чанки + tqdm дают тот же результат и видимый прогресс
BATCH_SIZE = 2048
ids_parts, scores_parts = [], []
for start in tqdm.tqdm(range(0, len(user_ids_encoded), BATCH_SIZE), desc='ALS recommend'):
    batch = user_ids_encoded[start:start + BATCH_SIZE]
    batch_ids, batch_scores = als_model.recommend(
        batch,
        user_item_matrix_train[batch],
        filter_already_liked_items=True,
        N=1,
    )
    ids_parts.append(batch_ids)
    scores_parts.append(batch_scores)

# Контракт прежний (tuple), downstream-код менять не нужно
als_recommendations = (
    np.concatenate(ids_parts, axis=0),
    np.concatenate(scores_parts, axis=0),
)


In [ ]:
# преобразуем полученные рекомендации в табличный формат
item_ids_enc = als_recommendations[0]
als_scores = als_recommendations[1]

als_recommendations = pd.DataFrame({
    "user_id_enc": user_ids_encoded,
    "item_id_enc": item_ids_enc.tolist(),
    "score": als_scores.tolist()})

# Разворачиваем (exploding) столбцы item_id_enc и score в отдельные строки.
# ignore_index=True сбрасывает индекс результирующего DataFrame,
# чтобы индексы были последовательными
als_recommendations = als_recommendations.explode(["item_id_enc", "score"], ignore_index=True)

# приводим типы данных
als_recommendations["item_id_enc"] = als_recommendations["item_id_enc"].astype("int")
als_recommendations["score"] = als_recommendations["score"].astype("float")
inv_product_map = {v: k for k, v in product_map.items()}
inv_user_map = {v: k for k, v in user_map.items()}
# получаем изначальные идентификаторы
als_recommendations["ncodpers"] = als_recommendations["user_id_enc"].map(inv_user_map)
als_recommendations["product"] = als_recommendations["item_id_enc"].map(inv_product_map)
als_recommendations = als_recommendations.drop(columns=["user_id_enc", "item_id_enc"])


In [ ]:
common_users_list = [int(x) for x in common_users]
filtered_events_test = common_test[common_test['ncodpers'].isin(common_users_list)]


In [ ]:
recommended_products_list = []
user_ids_list = []
non_common_users = list(filtered_events_test['ncodpers'].unique())
for user in non_common_users:
    user_ids_list.extend([user])
    recommended_products_list.append(np.nan)
recommended_tracks_df = pd.DataFrame({'ncodpers': user_ids_list,
                                      'product': recommended_products_list})
als_recs = als_recommendations[['ncodpers', 'product']]
recommended_tracks_df = pd.concat([recommended_tracks_df, als_recs], axis=0)
recommended_tracks_df.rename(columns={'product': 'recommended_product_id'}, inplace=True)
# Один клиент мог попасть и в NaN-строки, и в ALS-рекомендации: при дедупе
# оставляем непустую рекомендацию (NaN сортируется последним, keep='first'
# берёт непустое). Иначе lookup в сервисе всегда находил бы NaN → 0.
recommended_tracks_df = recommended_tracks_df.sort_values('recommended_product_id')
recommended_tracks_df = recommended_tracks_df.drop_duplicates('ncodpers', keep='first')
print(f'Клиентов с персональной рекомендацией: '
      f"{int(recommended_tracks_df['recommended_product_id'].notna().sum())} "
      f'из {len(recommended_tracks_df)}')
recommended_tracks_df.set_index('ncodpers').to_parquet('fastapi/personal_als.parquet')


### Качество ALS: precision@k / recall@k в двух скоупах

Раньше ALS использовался только как источник признака `recommended_product_id`, и качество
самих рекомендаций нигде не измерялось (CODE_REVIEW §3.7). Считаем precision@k / recall@k
на отложенном периоде 2016 года: рекомендации строятся по истории 2015 года, релевантными
считаются продукты, которые клиент действительно купил в 2016 году.

Скоупы:
- `common_users_2015_2016` — только клиенты, присутствовавшие в обоих годах
  (старый скоуп; оптимистичен: сюда не попадают cold-start клиенты 2016 года);
- `all_users_2016` — все клиенты с событиями 2016 года; для клиентов без истории
  2015 года персональных рекомендаций нет (покупок не предсказано → hits = 0).
  Это честная оценка системы на реальном потоке.

In [ ]:
def decode_item(item_index):
    """Индекс колонки ALS-матрицы -> имя продукта.

    Колонки матрицы построены по кодам продуктов (1..24), колонка 0 — служебная,
    поэтому products[idx - 1], а не products[idx].
    """
    idx = int(item_index)
    if 1 <= idx <= len(products):
        return products[idx - 1]
    return None


In [ ]:
# --- Оценка качества ALS на отложенном периоде (CODE_REVIEW §P1.11) ---
K = 5

als_top_k = als_model.recommend(
    user_ids_encoded,
    user_item_matrix_train[user_ids_encoded],
    filter_already_liked_items=True,
    N=K
)

recommendations_df = pd.DataFrame({
    'ncodpers': [inv_user_map[int(user_id)] for user_id in user_ids_encoded],
    'recommended': [
        [name for name in (decode_item(i) for i in items) if name is not None]
        for items in als_top_k[0]
    ],
})

# Релевантные продукты: все покупки клиента в 2016 году — для всех пользователей
# с событиями 2016 года, а не только common (в этом разница скоупов)
relevant_all = (
    common_test
    .groupby('ncodpers')['purchase']
    .apply(lambda codes: {products[int(code) - 1] for code in codes})
)

# Клиенты 2016 года без истории 2015 (cold start): персональных рекомендаций нет
cold_start_users = [int(user) for user in relevant_all.index
                    if int(user) not in set(recommendations_df['ncodpers'])]
eval_df = pd.concat([
    recommendations_df,
    pd.DataFrame({'ncodpers': cold_start_users,
                  'recommended': [[] for _ in cold_start_users]}),
], ignore_index=True)
eval_df['relevant'] = (
    eval_df['ncodpers'].map(relevant_all)
    .apply(lambda items: items if isinstance(items, set) else set())
)
eval_df['hits'] = [
    len(set(recommended) & relevant)
    for recommended, relevant in zip(eval_df['recommended'], eval_df['relevant'])
]
eval_df['relevant_size'] = eval_df['relevant'].apply(len)

os.makedirs('artifacts', exist_ok=True)
als_metric_rows = []
for scope, mask in [('common_users_2015_2016',
                     eval_df['ncodpers'].isin(set(recommendations_df['ncodpers']))),
                    ('all_users_2016', eval_df['ncodpers'].notna())]:
    scope_df = eval_df[mask]
    with_purchases = scope_df['relevant_size'] > 0
    als_metric_rows.append({
        'scope': scope,
        'k': K,
        'users': len(scope_df),
        'precision_at_k': float((scope_df['hits'] / K).mean()),
        'recall_at_k': float(
            (scope_df.loc[with_purchases, 'hits']
             / scope_df.loc[with_purchases, 'relevant_size']).mean()),
        'hit_rate': float((scope_df['hits'] > 0).mean()),
        'relevant_per_user': float(scope_df['relevant_size'].mean()),
    })
als_metrics = pd.DataFrame(als_metric_rows)
als_metrics.to_csv('artifacts/als_metrics.csv', index=False)
print(als_metrics.to_string(index=False))
print()
share_with_purchases = (eval_df['relevant_size'] > 0).mean()
print(f'Бейзлайн: у {share_with_purchases:.2%} клиентов в 2016 году была хотя бы одна покупка')
print('Скоуп all_users_2016 — честная оценка: cold-start клиентам без истории 2015 года '
      'ALS порекомендовать ничего не может. precision@k сравниваем с бейзлайном '
      'и с качеством классификатора.')


# Теперь делаем рекомендации на основе данных о клиенте

In [ ]:
# suppress warnings
warnings.filterwarnings('ignore')

# Фильтрация данных за 2015 год
df2015 = df[df['fecha_dato'] <= '2015-12-31']

# Получаем индексы последних записей для каждого клиента
last_idx = df2015.groupby('ncodpers')['fecha_dato'].idxmax()

# Собираем финальный датасет за один проход.
# fecha_dato среза СОХРАНЯЕМ: по нему строится временное разбиение train/test
# (удаляется после сплита). age/fecha_alta теперь не удаляем здесь — сырые
# значения нужны предобработке (fit считает по ним бины и когорты), а из
# финальной матрицы признаков их отфильтрует BASE_COLUMNS.
train_df = df2015.loc[last_idx].reset_index(drop=True)

train_df.head()


In [ ]:
train_df.to_csv('data/train_final.csv', index=False)


In [ ]:
first_buy = filtered_events_test.groupby('ncodpers')['fecha_dato'].idxmin()
first_buy_df = filtered_events_test.loc[first_buy, ['ncodpers', 'purchase']].copy()

# Продукты вне шорт-листа сворачиваем в 'other': иначе в таргете появляются классы
# с support = 1-2 объекта, а метрики по ним ничего не значат (CODE_REVIEW §3.1, §3.5)
first_buy_df['purchase'] = first_buy_df['purchase'].where(
    first_buy_df['purchase'].isin(target_codes), OTHER_CLASS
)

full_df = pd.merge(train_df, first_buy_df, on='ncodpers', how='left')

# 0 — «в 2016 году клиент ничего не купил»
full_df['purchase'] = full_df['purchase'].fillna(0).astype(int)

# Классы с единичными объектами делают сплит и метрики бессмысленными —
# сворачиваем их в 'other' (CODE_REVIEW §3.5)
MIN_CLASS_SUPPORT = 100
class_counts = full_df['purchase'].value_counts()
rare_classes = class_counts[(class_counts < MIN_CLASS_SUPPORT) & (class_counts.index != 0)].index
if len(rare_classes):
    print('Свернуты в other из-за малого support:',
          {int(code): label_map.get(int(code)) for code in rare_classes})
    full_df['purchase'] = full_df['purchase'].where(
        ~full_df['purchase'].isin(rare_classes), OTHER_CLASS
    )

print('Распределение классов (0 = покупки не было, {} = other):'.format(OTHER_CLASS))
print(full_df['purchase'].map(label_map).value_counts())


In [ ]:
full_df = pd.merge(full_df, recommended_tracks_df, on='ncodpers', how='left')

n_without_recs = int(full_df['recommended_product_id'].isna().sum())
print(f'Без персональной ALS-рекомендации: {n_without_recs:,} '
      f'({n_without_recs / len(full_df):.1%}) — для них признак = 0')

full_df['recommended_product_id'] = full_df['recommended_product_id'].fillna('0')


Итак, набор данных для обучения модели готов. Целевая переменная — колонка `purchase`:
код первого купленного в 2016 году продукта **из шорт-листа**, `0`, если клиент ничего
не купил, `other` (`99`), если первым куплен редкий продукт.

Почему шорт-лист, а не все 24 продукта: у большинства продуктов доля покупок меньше
процента, и классы по ним состояли бы из одного-двух объектов. Состав шорт-листа и правило
его отбора печатаются в разделе «Целевая переменная», туда же смотрит
`fastapi/preprocessing_params.json` (ключ `label_map`).

### Временное разбиение train/test

Раньше разбиение было стратифицированным (случайным) — метрики получались оптимистичнее
честного временного сценария (CODE_REVIEW §3.4). Заменяем его на **временное**, но важно —
осмысленное:

- Разбивать по **дате среза** (последнему `fecha_dato` клиента) нельзя: клиенты с последним
  срезом, скажем, в мае 2015 — это те, кто **ушёл из банка** до конца года, и «покупок
  в 2016» у них нет априори. Такой train оказывается почти целиком из бывших клиентов
  с таргетом 0 — вырожденное, смещённое разбиение (проверено экспериментально).
- Разбиваем по **дате привлечения клиента (`fecha_alta`)**: train — клиенты, привлечённые
  до cutoff-даты (≈70%), test — более новые клиенты. Сплит по времени привлечения не зависит
  от исхода 2016 года, профили одного клиента не смешиваются, а задача остаётся прежней
  («предсказать первую покупку 2016 года»). По сути это ответ на вопрос «обобщится ли
  модель, обученная на старой базе, на новых клиентах».

### Предобработка без утечек

Медианы, моды, квантили клиппинга, возрастные бины, когорты дат и наборы редких
категорий **обучаются только на train** (`fit_preprocessing_params`) — раньше они
считались на всём датасете до сплита (умеренная утечка, CODE_REVIEW §3.4).
Обе части затем трансформируются одной функцией `apply_preprocessing` — тем же
кодом, что использует сервис (`prepare_features`), поэтому train/serve skew
исключён по построению.

In [ ]:
# Временное разбиение по дате привлечения клиента (preprocessing.temporal_split):
# train — привлечённые до cutoff, test — новее. По дате среза (fecha_dato) делить
# нельзя: последний срез раного клиента означает отток из банка (см. markdown выше).
cutoff, train_mask, test_mask = temporal_split(
    pd.to_datetime(full_df['fecha_alta'], errors='coerce'), test_size=0.3)

train_raw = full_df[train_mask].reset_index(drop=True)
test_raw = full_df[test_mask].reset_index(drop=True)

# Клиент — одна строка; разбиение гарантирует, что профили одного клиента
# не встречаются в обеих частях (проверяем явно)
assert not set(train_raw['ncodpers']) & set(test_raw['ncodpers']), 'пересечение клиентов'

print(f'cutoff (fecha_alta): {cutoff.date()}')
print(f'train: {len(train_raw):,} строк, fecha_alta '
      f'{train_raw["fecha_alta"].min()} — {train_raw["fecha_alta"].max()}')
print(f'test:  {len(test_raw):,} строк, fecha_alta '
      f'{test_raw["fecha_alta"].min()} — {test_raw["fecha_alta"].max()}')
print(f'доля test: {len(test_raw) / len(full_df):.1%}')

y_train = train_raw['purchase']
y_test = test_raw['purchase']
print('Классы в train:', y_train.value_counts().to_dict())
print('Классы в test:', y_test.value_counts().to_dict())

# fecha_dato и идентификатор в признаки не входят; таргет отделяем от признаков
X_train_raw = train_raw.drop(columns=['purchase', 'fecha_dato', 'ncodpers'])
X_test_raw = test_raw.drop(columns=['purchase', 'fecha_dato', 'ncodpers'])


In [ ]:
# Статистики предобработки — ТОЛЬКО на train (CODE_REVIEW §3.4):
# медианы, моды, возрастные бины, когорты дат, редкие категории, квантили
# клиппинга и эталонные гистограммы дрейфа (для мониторинга в сервисе).
params = fit_preprocessing_params(X_train_raw)

print('Медианы train:', params.medians)
print('Границы клиппинга train:', params.clip_bounds)
print('Возрастные интервалы train:', params.age_intervals)
print('Когорты дат:', {col: len(spec.get('intervals', ()))
                       for col, spec in params.date_cohorts.items()})
print(f'Сколько колонок с редкими категориями: '
      f'{sum(bool(values) for values in params.replacer.values())}')
print('Эталон дрейфа:', sorted(params.drift_reference or {}))


In [ ]:
# Трансформация обеих частей параметрами train — тем же кодом, что на проде.
# Рекомендация ALS уже подмёржена (recommended_product_id), на проде её
# подставляет lookup по ncodpers — дальше путь общий.
X_train = apply_preprocessing(X_train_raw, params)
X_test = apply_preprocessing(X_test_raw, params)

# Матрица признаков — ровно базовые колонки модели (как в сервисе)
X_train = X_train[BASE_COLUMNS]
X_test = X_test[BASE_COLUMNS]
print(f'train: {X_train.shape}, test: {X_test.shape}')


### Синтез новых признаков

In [ ]:
# Генерация признаков — из общего модуля предобработки (CODE_REVIEW §P2.14):
# обучение и сервис используют одни и те же функции.
# Агрегаты (mean_renta_by_pais_residencia, median_antiguedad_by_segmento и др.)
# считаются ТОЛЬКО на train и затем применяются к test (раньше считались на
# всём датасете до сплита — утечка, CODE_REVIEW §3.4); те же агрегаты уезжают
# в fastapi/preprocessing_params.json и используются сервисом.
X_train['antiguedad'] = X_train['antiguedad'].astype(int)
X_test['antiguedad'] = X_test['antiguedad'].astype(int)

df_train, aggregates = feature_engineering(X_train)
df_test, _ = feature_engineering(X_test, aggregates)


In [ ]:
# Автоматическое определение количественных и категориальных переменных —
# по TRAIN: категориальные — это нечисловые колонки (в них после сворачивания
# редких категорий появляется 'other') и колонки с малым числом уникальных
# значений. Те же списки применяются к test и уезжают в артефакт сервиса.
numeric_features = []
categorical_features = []

for column in df_train.columns:
    unique_values = df_train[column].nunique()
    if unique_values < 25 or not pd.api.types.is_numeric_dtype(df_train[column]):
        categorical_features.append(column)
    else:
        numeric_features.append(column)

print(f'Numeric features ({len(numeric_features)}):', numeric_features)
print(f'Categorical features ({len(categorical_features)}):', categorical_features)

for cat in tqdm.tqdm(categorical_features, desc='Converting categorical features to string'):
    df_train[cat] = df_train[cat].astype('str')
    df_test[cat] = df_test[cat].astype('str')

# Контроль остаточных пропусков (после fillna/агрегатов их быть не должно)
n_nan = int(df_train.isna().sum().sum() + df_test.isna().sum().sum())
print(f'NaN в матрицах признаков: {n_nan}')


In [ ]:
# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


In [ ]:
load_dotenv()
TRACKING_SERVER_HOST = os.getenv('MLFLOW_TRACKING_HOST', '127.0.0.1')
TRACKING_SERVER_PORT = int(os.getenv('MLFLOW_TRACKING_PORT', '5000'))

EXPERIMENT_NAME = "RecSys_Modeling"
RUN_NAME = "fit"

FS_ASSETS = 'modeling'
os.makedirs(FS_ASSETS, exist_ok=True)

# Локальный tracking-сервер MLflow
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

mlflow.set_experiment(EXPERIMENT_NAME)


In [ ]:
def macro_roc_auc(y_true, y_proba, classes):
    """ROC-AUC (ovr, macro) по классам, которые есть в y_true.

    В выборке может не быть части классов (редкие покупки), а sklearn в этом случае
    падает на несовпадении числа классов и колонок вероятностей — отсутствующие
    классы исключаем из усреднения.
    """
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    present = [index for index, cls in enumerate(classes) if (y_true == cls).any()]
    if len(present) < 2:
        return float('nan')
    if len(present) == 2:
        # в бинарном случае sklearn ждет одномерный вектор вероятностей
        return roc_auc_score((y_true == classes[present[1]]).astype(int), y_proba[:, present[1]])
    return roc_auc_score(
        y_true,
        y_proba[:, present],
        multi_class='ovr',
        average='macro',
        labels=[classes[index] for index in present],
    )


def cross_val_roc_auc(pipeline, X_train, y_train, cv):
    """ROC-AUC по фолдам кросс-валидации — тестовая выборка не используется."""
    scores = []
    for train_index, valid_index in cv.split(X_train, y_train):
        fold_model = clone(pipeline).fit(X_train.iloc[train_index], y_train.iloc[train_index])
        scores.append(macro_roc_auc(
            y_train.iloc[valid_index].to_numpy(),
            fold_model.predict_proba(X_train.iloc[valid_index]),
            list(fold_model.classes_),
        ))
    return np.asarray(scores, dtype=float)


def make_objective(pipeline, X_train, y_train, cv):
    """Objective для Optuna: ROC-AUC по кросс-валидации на train.

    Тестовая выборка в подборе гиперпараметров не участвует (CODE_REVIEW §3.3):
    иначе метрика на тесте — это максимум из перебора, а не честная оценка.
    """

    def objective(trial):
        # Извлечение параметров
        params_trial = {
            'classifier__n_estimators': trial.suggest_int('classifier__n_estimators', 5, 50),
            'classifier__max_depth': trial.suggest_categorical(
                'classifier__max_depth', [1, 5, 10, 30, None]
            ),
            'classifier__min_samples_split': trial.suggest_int(
                'classifier__min_samples_split', 2, 20
            ),
        }
        pipeline.set_params(**params_trial)

        # Оценка — только по фолдам внутри train
        scores = cross_val_roc_auc(pipeline, X_train, y_train, cv)
        trial.set_user_attr('cv_std', float(np.nanstd(scores)))
        return float(np.nanmean(scores))

    return objective


In [ ]:
with mlflow.start_run(run_name=RUN_NAME):
    # Разбиение уже выполнено выше — временное, по дате среза клиента (CODE_REVIEW §3.4)
    X_train_ml, X_test_ml = df_train, df_test
    logger.info("Data loaded and preprocessed.")
    print(f'train: {X_train_ml.shape}, test: {X_test_ml.shape}')

    # Параметры разбиения и обучения — в MLflow (воспроизводимость)
    mlflow.log_param('split_type', 'temporal_fecha_alta')
    mlflow.log_param('split_cutoff', str(cutoff.date()))
    mlflow.log_param('split_test_size', 0.3)
    mlflow.log_param('preprocessing_fit', 'train_only')

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ])
    logger.info("Preprocessor initialized.")

    # Пайплайн с учетом дисбаланса классов
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('feature_selection', SelectFromModel(
            RandomForestClassifier(n_estimators=50, class_weight='balanced', random_state=0)
        )),
        ('classifier', RandomForestClassifier(class_weight='balanced', random_state=0))
    ])
    logger.info("Pipeline created with class weight handling.")

    # Поиск гиперпараметров с Optuna: скоринг по кросс-валидации на train,
    # тестовая выборка используется один раз — для финальной оценки (CODE_REVIEW §P1.9)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(make_objective(pipeline, X_train_ml, y_train, cv), n_trials=20, timeout=1200)

    completed_trials = [trial for trial in study.trials
                        if trial.state == optuna.trial.TrialState.COMPLETE]
    if not completed_trials:
        logger.error("Optuna failed to complete any trials within time limit.")
        raise ValueError("No Optuna trials completed")

    # Логирование лучших параметров
    best_params = study.best_params
    mlflow.log_params(best_params)
    mlflow.log_metric('cv_roc_auc_mean', study.best_value)
    mlflow.log_metric('cv_roc_auc_std', study.best_trial.user_attrs.get('cv_std', float('nan')))
    mlflow.log_metric('optuna_trials', len(study.trials))
    print(f'Optuna: {len(study.trials)} триалов, лучший CV ROC-AUC = {study.best_value:.4f}')
    print('Лучшие параметры:', best_params)

    # Обучение финальной модели на всем train
    final_model = pipeline.set_params(**best_params).fit(X_train_ml, y_train)
    logger.info("Final model trained with best parameters.")

    # --- Оценка: тест трогаем один раз, после выбора гиперпараметров --------
    y_pred = final_model.predict(X_test_ml)
    y_proba = final_model.predict_proba(X_test_ml)
    classes = [int(cls) for cls in final_model.named_steps['classifier'].classes_]

    # Взвешенные метрики считаем для сопоставимости с прошлыми запусками,
    # но основной вес в них у класса «покупки не было» (CODE_REVIEW §P1.11)
    metrics = {
        'roc_auc_ovr_macro': macro_roc_auc(y_test, y_proba, classes),
        'accuracy': accuracy_score(y_test, y_pred),
        'precision_weighted': precision_score(y_test, y_pred, average='weighted',
                                              zero_division=0),
        'recall_weighted': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1_weighted': f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'precision_macro': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
    }

    # PR-AUC (average precision) по каждому классу: для сильного дисбаланса
    # это честнее ROC-AUC, а базовый уровень случайного ранжирования = доля класса
    pr_auc_rows = []
    for index, cls in enumerate(classes):
        y_binary = (y_test.to_numpy() == cls).astype(int)
        if y_binary.sum() == 0:
            continue
        pr_auc_rows.append({
            'class': cls,
            'product': label_map.get(cls, str(cls)),
            'support': int(y_binary.sum()),
            'share': float(y_binary.mean()),
            'pr_auc': float(average_precision_score(y_binary, y_proba[:, index])),
            'roc_auc_ovr': float(roc_auc_score(y_binary, y_proba[:, index])),
        })
    pr_auc_df = pd.DataFrame(pr_auc_rows).sort_values('share', ascending=False)
    metrics['pr_auc_macro'] = float(pr_auc_df['pr_auc'].mean())

    mlflow.log_metrics(metrics)
    for row in pr_auc_rows:
        mlflow.log_metric(f"pr_auc_class_{row['class']}", row['pr_auc'])
        mlflow.log_metric(f"support_class_{row['class']}", row['support'])
    logger.info("Model metrics logged.")

    mlflow.sklearn.log_model(final_model, "model")
    logger.info("Best model logged to MLflow.")
    training_run_id = mlflow.active_run().info.run_id
    print(f'MLflow run_id: {training_run_id}')

    # Отчет о качестве: per-class таблица + macro + PR-AUC (CODE_REVIEW §P1.11)
    report = classification_report(y_test, y_pred, digits=4, zero_division=0)
    with open('artifacts/classification_report.txt', 'w', encoding='utf-8') as f:
        f.write('Отчет о качестве последнего обучающего запуска (modeling.ipynb)\n')
        f.write(f'Целевая переменная: первый купленный продукт 2016 года из шорт-листа; '
                f'классы: {label_map}\n')
        f.write('Разбиение: временное по дате привлечения клиента (fecha_alta), '
                f'cutoff {cutoff.date()}; '
                f'train: {X_train_ml.shape}, test: {X_test_ml.shape}\n')
        f.write('Статистики предобработки и агрегаты обучены только на train; '
                'гиперпараметры выбраны по CV на train, тест использован один раз\n\n')
        f.write('Взвешенные метрики (доминирует класс "покупки не было"):\n')
        for name, value in metrics.items():
            f.write(f'  {name}: {value:.4f}\n')
        f.write('\n' + report)
        f.write('\nPR-AUC по классам (случайное ранжирование дает PR-AUC = доля класса):\n')
        f.write(pr_auc_df.to_string(index=False) + '\n')
    mlflow.log_artifact('artifacts/classification_report.txt')
    logger.info("Classification report logged to MLflow.")
    print('\n' + report)
    print(pr_auc_df.to_string(index=False))

    # Feature importance
    sup = final_model.named_steps['feature_selection'].get_support()
    fname = final_model.named_steps['preprocessor'].get_feature_names_out()
    fi_df = pd.DataFrame({'feature': fname, 'support': sup})
    fi_df = fi_df[fi_df['support']]
    fi_df['importance'] = final_model.named_steps['classifier'].feature_importances_
    fi_df = fi_df.sort_values(by='importance', ascending=False)
    fi_df.to_csv('artifacts/feature_importances.csv', index=False)
    mlflow.log_artifact('artifacts/feature_importances.csv')
    logger.info("Feature importances logged to MLflow.")

    if os.path.exists('artifacts/als_metrics.csv'):
        mlflow.log_artifact('artifacts/als_metrics.csv')

    logger.info(f"Метрики на тесте: ROC-AUC (ovr macro) = {metrics['roc_auc_ovr_macro']:.4f}, "
                f"F1 macro = {metrics['f1_macro']:.4f}, "
                f"PR-AUC macro = {metrics['pr_auc_macro']:.4f}")
    logger.info("Experiment completed successfully!")


ВЫВОД: Модель обучена и сохранена в MLFLOW

### Артефакт параметров предобработки

Всё, что нужно для предсказания на проде (медианы, моды, границы клиппинга, возрастные
бины, когорты дат, агрегаты, словарь редких категорий, расшифровка классов, эталонные
гистограммы дрейфа для мониторинга), сохраняется в `fastapi/preprocessing_params.json`.
Сервис `app1.py` читает этот файл, поэтому предобработка на проде совпадает с обучением
и переобучение не ломает сервис (CODE_REVIEW §P1.10).

Статистики обучены **только на train** (CODE_REVIEW §3.4); в блоке `split`
зафиксированы тип разбиения и cutoff-дата.

In [ ]:
def json_converter(obj):
    """Конвертер numpy/pandas-типов в JSON для артефактов сервиса."""
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, pd.Timestamp):
        return obj.strftime('%Y-%m-%d')
    if isinstance(obj, pd.Interval):
        return [float(obj.left), float(obj.right)]
    raise TypeError(f"Object of type {obj.__class__.__name__} is not JSON serializable")


def save_json(path, payload):
    """Сохраняет артефакт предобработки для микросервиса."""
    with open(path, 'w', encoding='utf-8') as file:
        json.dump(payload, file, indent=2, ensure_ascii=False, default=json_converter)
    print(f'Сохранено: {path}')


preprocessing_params = {
    'created_at': pd.Timestamp.now().isoformat(timespec='seconds'),
    'dataset': 'data/train_ver2.csv',
    'target': {
        'definition': 'первый продукт, купленный клиентом в 2016 году, из шорт-листа',
        'shortlist_rule': f'доля покупок 2015 года >= {MIN_PURCHASE_SHARE:.0%}',
        'classes': {str(code): name for code, name in label_map.items()},
    },
    'split': {
        'type': 'temporal_fecha_alta',
        'cutoff': str(cutoff.date()),
        'test_size': 0.3,
        'definition': 'train: клиенты, привлечённые до cutoff (fecha_alta); test: новее',
        'fit': 'статистики предобработки и агрегаты обучены только на train',
    },
    'medians': params.medians,
    'modes': params.modes,
    'clip_bounds': params.clip_bounds,
    'age_intervals': params.age_intervals,
    'date_cohorts': params.date_cohorts,
    'aggregates': aggregates,
    'replacer': params.replacer,
    'drift_reference': params.drift_reference,
    'label_map': {str(code): name for code, name in label_map.items()},
    'feature_columns': {
        'numeric': numeric_features,
        'categorical': categorical_features,
    },
}

save_json('fastapi/preprocessing_params.json', preprocessing_params)
save_json('fastapi/replacer.json', params.replacer)
print('Классы модели:', preprocessing_params['label_map'])
print('Агрегаты для сервиса:', list(aggregates))
print('Эталон дрейфа:', sorted(params.drift_reference or {}))


In [ ]:
# Канонический экспорт модели для микросервиса (CODE_REVIEW §P2.20):
# sklearn-пайплайн целиком — сервис грузит его через joblib.load.
# Рядом пишем fastapi/model_version.json (run id, дата, метрики, классы,
# схема разбиения), чтобы версия модели была зафиксирована, а не только в MLflow.
joblib.dump(final_model, 'fastapi/saved_model.pkl')
print('Модель сохранена: fastapi/saved_model.pkl')

model_version = {
    'created_at': pd.Timestamp.now().isoformat(timespec='seconds'),
    'model_path': 'fastapi/saved_model.pkl',
    'mlflow_run_id': training_run_id if 'training_run_id' in dir() else None,
    'experiment': EXPERIMENT_NAME,
    'dataset': 'data/train_ver2.csv',
    'target': preprocessing_params['target'],
    'split': preprocessing_params['split'],
    'train_shape': list(X_train_ml.shape),
    'test_shape': list(X_test_ml.shape),
    'best_params': best_params,
    'cv_roc_auc_mean': float(study.best_value),
    'metrics': {name: round(float(value), 4) for name, value in metrics.items()},
    'label_map': {str(code): name for code, name in label_map.items()},
}
save_json('fastapi/model_version.json', model_version)
print('Версия зафиксирована: fastapi/model_version.json',
      f"(run_id={model_version['mlflow_run_id']})")


# 📊 Анализ метрик рекомендательной системы

---

## Методология оценки (актуальный запуск)

- **Разбиение train/test — временное**: в test попали клиенты с самыми поздними
  срезами 2015 года (после cutoff-даты, см. ячейку разбиения). Профили одного
  клиента не смешиваются между частями — метрики больше не завышены за счёт
  «подглядывания» в будущее.
- **Статистики предобработки и агрегаты обучены только на train** — утечки
  до сплита больше нет.
- **Качество ALS** (`artifacts/als_metrics.csv`) — в двух скоупах: на клиентах
  обоих годов и на всех покупателях 2016 года (включая cold-start).

Числа в этой тетрадке и в отчётах — артефакты **последнего прогона**; при
переобучении они обновляются, поэтому конкретные значения смотрите в
`artifacts/classification_report.txt` и выводе ячеек выше.

---

## Как читать метрики

> ⚠️ **Взвешенные метрики — это в основном метрика класса «ничего не купил»
> (доля ~80% выборки), а не качество рекомендаций.**

### Честная картина — по macro и PR-AUC:

- `f1_macro`, `precision_macro`, `recall_macro` — равноправное усреднение по классам;
- `pr_auc_macro` — при доле покупок ~0.01–0.06 по классам (для случайного
  ранжирования PR-AUC равен доле класса, колонка `share` в отчёте).

**Типичная картина:** модель уверенно отсеивает тех, кому ничего не нужно
(high precision по классу 0), угадывает, что клиент *что-то* купит, но редко
попадает точно в продукт. Для cold-start cross-sell на 24 банковских продуктах
это рабочая точка. По временному разбиению метрики, как правило, немного ниже,
чем по стратифицированному, — это и есть честная цена обобщения.

### Что смотреть в per-class таблице:

- 🌟 클래스-пары (`ind_nom_pens_ult1` после `ind_nomina_ult1`) обычно дают
  самый высокий lift — почти детерминированная связка;
- 📉 массовые продукты с низкой повторной покупкой (`ind_recibo_ult1`)
  традиционно самые трудные — их лифт невелик;
- 🔬 у редких классов (support в сотни на сотни тысяч теста) большой lift —
  в первую очередь шум: ориентируйтесь на support и macro-метрики.